#Imports

In [ ]:
import re
import json

from dataclasses import dataclass
from enum import Enum, auto
from collections import Counter

from datasets import load_dataset

#Load Data Set

In [ ]:
DATASET_NAME = "Helsinki-NLP/un_pc"
CONFIG_NAME = "ar-en"

MAX_SAMPLES = 160000000

OUTPUT_FILE = "paragraphs.jsonl"

In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    CONFIG_NAME,
    split="train",
    streaming=True,
)

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
samples = [sample for _, sample in zip(range(25), dataset)]

with open("first_25_samples.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=4, ensure_ascii=False)

In [ ]:
from itertools import islice

MAX_SAMPLES = 1600000000
TAIL_SAMPLES = 10000

START_IDX = MAX_SAMPLES - TAIL_SAMPLES  # 1,500,000

dataset = list(islice(dataset, START_IDX, MAX_SAMPLES))

#Preparing for cleaning

In [ ]:
class Role(Enum):

    CONTENT = auto()

    BOUNDARY = auto()  #Header

    STRUCTURE = auto() #EXTRAS LIKE NUMBER OF DOCUMENT AND SERIAL NUMBERS -> Meta data

In [ ]:
@dataclass(slots=True)
class Sample:

    index: int

    en: str

    ar: str

@dataclass
class ParseResult:

    sample: Sample

    line_type: str

    role: Role

Meta Data

In [ ]:
DOCUMENT_CODE_RE = re.compile(
    r"^[A-Z]{1,3}(?:/[A-Z0-9.-]+)+$"
)

DATE_RE = re.compile(
    r"^\d{1,2}\s+[A-Za-z]+(?:-\d{1,2}\s+[A-Za-z]+)?\s+\d{4}$"
)

ROMAN_SECTION_RE = re.compile(
    r"^[IVXLCDM]+\.\s+"
)

TOC_RE = re.compile(
    r"^.+\.\s*\d+$"
)

FOOTNOTE_RE = re.compile(
    r"^\d+/"
)

SESSION_RE = re.compile(
    r"^\w+(?:-\w+)?\s+session$",
    re.IGNORECASE
)

WORD_RE = re.compile(
    r"[A-Za-z]+(?:[-'][A-Za-z]+)*"
)

DATE_PREFIX_RE = re.compile(
    r"^\d{1,2}\s+[A-Za-z]+\s+\d{4}:?\s+[^.]+$"
)


In [ ]:
HISTORIC_CONTEXT_RE = re.compile(
    r"^\[\d{1,2}\s+[A-Za-z]+\s+\d{4}\]\s+"
)

In [ ]:
KNOWN_METADATA = {


    "GENERAL",

    "CONTENTS",

    "ANNEX",

    "ANNEXES",

    "APPENDIX",

    "Page",

    "Distr.",

}

In [ ]:
def normalize(text):

    return " ".join(text.strip().split())


def has_lowercase(text):

    return any(c.islower() for c in text)


WOORD_RE = re.compile(
    r"[A-Za-z]{2,}(?:[-'][A-Za-z]{2,})*"
)

def word_count(text):

    return len(WOORD_RE.findall(text))


def ends_sentence(text):

    return text.endswith((".", "!", "?", ",", ";"))

In [ ]:
def stream_samples(dataset):

    for index, sample in enumerate(dataset):

        if index >= MAX_SAMPLES:
            break

        yield Sample(

            index=index,

            en=normalize(sample["translation"]["en"]),

            ar=normalize(sample["translation"]["ar"])

        )

In [ ]:
def is_document_code(text):

    return DOCUMENT_CODE_RE.fullmatch(text) is not None

In [ ]:
def is_metadata(text):

    if text in KNOWN_METADATA:
        return True
#only date
    if DATE_RE.fullmatch(text):
        return True

    if text.startswith("ORIGINAL:"):
        return True

    if text.startswith("[Original:"):
        return True

    return False

In [ ]:
def is_toc(text):

    return TOC_RE.fullmatch(text) is not None

In [ ]:
def is_footnote(text):

    return FOOTNOTE_RE.match(text) is not None

In [ ]:
def is_session(text):

    return SESSION_RE.fullmatch(text) is not None

In [ ]:
def is_header(text):

  if not has_lowercase(text):
      return True

  if word_count(text) <= 5:
      return True


  if word_count(text) <= 10 and not ends_sentence(text):
      return True

  return False

In [ ]:
def parse(sample):

    text = sample.en

    if not text:

        return ParseResult(
            sample=sample,
            line_type="EMPTY",
            role=Role.STRUCTURE,
        )

    if is_document_code(text):

        return ParseResult(
            sample=sample,
            line_type="DOCUMENT_CODE",
            role=Role.STRUCTURE,
        )

    if is_metadata(text):

        return ParseResult(
            sample=sample,
            line_type="METADATA",
            role=Role.STRUCTURE,
        )

    if is_toc(text):

        return ParseResult(
            sample=sample,
            line_type="TOC",
            role=Role.STRUCTURE,
        )

    if is_footnote(text):

        return ParseResult(
            sample=sample,
            line_type="FOOTNOTE",
            role=Role.STRUCTURE,
        )

    if is_session(text):

        return ParseResult(
            sample=sample,
            line_type="SESSION",
            role=Role.BOUNDARY,
        )

    if is_header(text):

        return ParseResult(
            sample=sample,
            line_type="HEADER",
            role=Role.BOUNDARY,
        )

    return ParseResult(
        sample=sample,
        line_type="BODY",
        role=Role.CONTENT,
    )

In [ ]:
def normalize_roles(results):

    context = {
        "document_code": [],
        "metadata": [],
        "historic_headline": [],
        "historic_context": [],
        "session": [],
        "section": [],
        "subsection": [],
        "header": [],
        "toc": [],
        "footnote": [],
    }

    pending = None

    for result in results:

        if result.role != Role.CONTENT:

            key = result.line_type.lower()

            if key in context:
                context[key].append(result.sample.en)

            if pending is None:

                pending = result.role

            elif pending == Role.STRUCTURE and result.role == Role.BOUNDARY:

                pending = Role.BOUNDARY

            continue

        if pending is not None:

            yield {
                "context": {
                    k: v.copy()
                    for k, v in context.items()
                },
                "result": ParseResult(
                    sample=result.sample,
                    line_type=pending.name,
                    role=pending,
                ),
            }

            for value in context.values():
                value.clear()

            pending = None

        yield {
            "context": {
                k: v.copy()
                for k, v in context.items()
            },
            "result": result,
        }

    if pending is not None:

        yield {
            "context": {
                k: v.copy()
                for k, v in context.items()
            },
            "result": ParseResult(
                sample=Sample(-1, "", ""),
                line_type=pending.name,
                role=pending,
            ),
        }

In [ ]:
counter = Counter()

examples = {}

for line_type in [
    "BODY",
    "HEADER",
    "SECTION",
    "SUBSECTION",
    "SESSION",
    "DOCUMENT_CODE",
    "METADATA",
    "TOC",
    "FOOTNOTE",
    "HISTORIC_HEADLINE",
    "HISTORIC_CONTEXT",
    "EMPTY",
]:
    examples[line_type] = []

debug = []

for sample in stream_samples(dataset):

    result = parse(sample)

    counter[result.line_type] += 1

    if (
        result.line_type in examples
        and
        len(examples[result.line_type]) < 50
    ):
        examples[result.line_type].append(sample.en)

    debug.append({
        "index": sample.index,
        "line_type": result.line_type,
        "role": result.role.name,
        "text": sample.en,
    })

In [ ]:
for line_type, rows in examples.items():

    print("=" * 80)
    print(line_type)
    print("=" * 80)

    for row in rows:
        print(row)

    print()

In [ ]:
@dataclass(slots=True)
class Paragraph:

    start_index: int

    end_index: int

    en: str

    ar: str

    num_samples: int

In [ ]:
class ParagraphBuffer:

    def __init__(self):

        self.samples = []

    def clear(self):

        self.samples.clear()

    def empty(self):

        return len(self.samples) == 0

    def append(self, sample):

        self.samples.append(sample)

    def build(self):

        if self.empty():
            return None

        paragraph = Paragraph(

            start_index=self.samples[0].index,

            end_index=self.samples[-1].index,

            en=" ".join(s.en for s in self.samples),

            ar=" ".join(s.ar for s in self.samples),

            num_samples=len(self.samples)

        )

        self.clear()

        return paragraph

In [ ]:
    def build(self):

        if self.empty():
            return None

        paragraph = Paragraph(

            start_index=self.samples[0].index,

            end_index=self.samples[-1].index,

            en=" ".join(s.en for s in self.samples),

            ar=" ".join(s.ar for s in self.samples),

            num_samples=len(self.samples)

        )

        self.clear()

        return paragraph

In [ ]:
def paragraph_stream(dataset):

    buffer = ParagraphBuffer()

    parsed = (
        parse(sample)
        for sample in stream_samples(dataset)
    )

    for item in normalize_roles(parsed):

        result = item["result"]

        if result.role == Role.CONTENT:

            buffer.append(result.sample)

        elif result.role == Role.BOUNDARY:

            paragraph = buffer.build()

            if paragraph is not None:
                yield paragraph

        elif result.role == Role.STRUCTURE:

            paragraph = buffer.build()

            if paragraph is not None:
                yield paragraph

            buffer.clear()

    paragraph = buffer.build()

    if paragraph is not None:
        yield paragraph

In [ ]:
paragraphs = list(paragraph_stream(dataset))

print(f"Paragraphs: {len(paragraphs)}")

sample_counts = [p.num_samples for p in paragraphs]

print("Average samples :", sum(sample_counts) / len(sample_counts))
print("Max samples     :", max(sample_counts))
print("Min samples     :", min(sample_counts))

In [ ]:
for i, p in enumerate(paragraphs[:10]):

    print("=" * 100)
    print(f"Paragraph {i+1}")
    print("=" * 100)

    print(f"Samples : {p.num_samples}")
    print(f"Indices : {p.start_index} -> {p.end_index}")
    print()

    print(p.en)
    print()

In [ ]:
paragraphs = paragraph_stream(dataset)

for i, paragraph in enumerate(paragraphs, 1):

    print("=" * 120)
    print(f"Paragraph {i}")
    print("=" * 120)

    print(f"Samples : {paragraph.num_samples}")
    print(f"Indices : {paragraph.start_index} -> {paragraph.end_index}")
    print()

    print("English:")
    print(paragraph.en)
    print()

    print("Arabic:")
    print(paragraph.ar)
    print()

    if i == 25:
        break

In [ ]:
import json

OUTPUT_JSON = "paragraph_samples.json"

paragraphs = []

for i, paragraph in enumerate(paragraph_stream(dataset)):

    paragraphs.append({
        "paragraph_id": i + 1,
        "num_samples": paragraph.num_samples,
        "start_index": paragraph.start_index,
        "end_index": paragraph.end_index,
        "english": paragraph.en,
        "arabic": paragraph.ar,
    })

    if i == 24:
        break

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        paragraphs,
        f,
        ensure_ascii=False,
        indent=4,
    )

print(f"Saved {len(paragraphs)} paragraphs to {OUTPUT_JSON}")

In [ ]:
import json

OUTPUT_JSON = "original_samples_0_413.json"

samples = []

for i, sample in enumerate(dataset):

    if i > 413:
        break

    samples.append({
        "index": i,
        "english": sample["translation"]["en"],
        "arabic": sample["translation"]["ar"],
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        samples,
        f,
        ensure_ascii=False,
        indent=4,
    )

print(f"Saved {len(samples)} samples to {OUTPUT_JSON}")

In [ ]:
import re

word_counts = []

for paragraph in paragraph_stream(dataset):

    words = re.findall(r"\b\w+\b", paragraph.en)
    word_counts.append(len(words))

print(f"Paragraphs      : {len(word_counts)}")
print(f"Average words   : {sum(word_counts) / len(word_counts):.2f}")
print(f"Median words    : {sorted(word_counts)[len(word_counts)//2]}")
print(f"Max words       : {max(word_counts)}")
print(f"Min words       : {min(word_counts)}")

print()
print("Distribution")
print(f"<100   : {sum(x < 100 for x in word_counts)}")
print(f"100-200: {sum(100 <= x < 200 for x in word_counts)}")
print(f"200-300: {sum(200 <= x < 300 for x in word_counts)}")
print(f"300-500: {sum(300 <= x < 500 for x in word_counts)}")
print(f">=500   : {sum(x >= 500 for x in word_counts)}")

In [ ]:
import re

word_counts = []

for paragraph in paragraph_stream(dataset):

    words = re.findall(r"\b\w+\b", paragraph.en)
    word_counts.append(len(words))

print(f"Paragraphs      : {len(word_counts)}")
print(f"Average words   : {sum(word_counts) / len(word_counts):.2f}")
print(f"Median words    : {sorted(word_counts)[len(word_counts)//2]}")
print(f"Max words       : {max(word_counts)}")
print(f"Min words       : {min(word_counts)}")

print()
print("Distribution")
print(f"<100   : {sum(x < 100 for x in word_counts)}")
print(f"<50   : {sum(x < 50 for x in word_counts)}")
print(f"<25  : {sum(x < 25 for x in word_counts)}")
print(f"<10   : {sum(x < 10 for x in word_counts)}")
print(f"<5  : {sum(x < 5 for x in word_counts)}")
print(f"<2  : {sum(x < 2 for x in word_counts)}")
print(f"100-200: {sum(100 <= x < 200 for x in word_counts)}")
print(f"200-300: {sum(200 <= x < 300 for x in word_counts)}")
print(f"300-500: {sum(300 <= x < 500 for x in word_counts)}")
print(f">=500   : {sum(x >= 500 for x in word_counts)}")
print(f">=1000   : {sum(x >= 500 for x in word_counts)}")

In [ ]:
import re

count = 0

for paragraph in paragraph_stream(dataset):

    words = re.findall(r"\b\w+\b", paragraph.en)

    if len(words) < 5:

        count += 1

        #print("=" * 120)
        print(f"Paragraph {count}")
        print(f"Words : {len(words)}")
        print(f"Samples: {paragraph.num_samples}")
        print()

        print("English:")
        print(paragraph.en)
        print()

        print("Arabic:")
        print(paragraph.ar)
        print()

print(f"\nTotal paragraphs with <5 words: {count}")

In [ ]:
import re

for i, paragraph in enumerate(paragraph_stream(dataset), 1):

    words = len(re.findall(r"\b\w+\b", paragraph.en))

    if words > 1000:

        print("=" * 100)
        print(f"Paragraph {i}")
        print(f"Words : {words}")
        print(f"Samples: {paragraph.num_samples}")
        print(f"Indices: {paragraph.start_index} -> {paragraph.end_index}")

In [ ]:
paragraph = next(
    p for p in paragraph_stream(dataset)
    if len(re.findall(r"\b\w+\b", p.en)) > 1000
)

print(paragraph.en[:5000])

In [ ]:
print(type(paragraphs[0]))

In [ ]:
paragraphs = list(paragraph_stream(dataset))

In [ ]:
import re

MAX_WORDS = 500

cleaned_paragraphs = []

removed = 0

for p in paragraphs:

    words = len(re.findall(r"\b\w+\b", p.en))

    if words <= MAX_WORDS:
        cleaned_paragraphs.append(p)
    else:
        removed += 1

print(f"Removed : {removed}")
print(f"Remaining : {len(cleaned_paragraphs)}")

paragraphs = cleaned_paragraphs

In [ ]:
import html
import re

for p in paragraphs:

    p.en = html.unescape(p.en)
    p.ar = html.unescape(p.ar)

    p.en = re.sub(r"\s+", " ", p.en).strip()
    p.ar = re.sub(r"\s+", " ", p.ar).strip()

print("HTML cleaning done.")

In [ ]:
import html
import re

ZERO_WIDTH_RE = re.compile(
    r"[\u200b-\u200f\u2060\ufeff]"
)

for p in paragraphs:

    p.en = html.unescape(p.en)
    p.ar = html.unescape(p.ar)

    p.en = p.en.replace("\u00ad", "")
    p.ar = p.ar.replace("\u00ad", "")

    p.en = ZERO_WIDTH_RE.sub("", p.en)
    p.ar = ZERO_WIDTH_RE.sub("", p.ar)

    p.en = re.sub(r"\s+", " ", p.en).strip()
    p.ar = re.sub(r"\s+", " ", p.ar).strip()

print("Cleaning finished.")

In [ ]:
count = 0

for p in paragraphs:

    if any(x in p.en for x in ["&quot;", "&apos;", "&amp;"]):
        count += 1

print(count)

In [ ]:
count = 0

for p in paragraphs:

    if any(x in p.ar for x in ["&quot;", "&apos;", "&amp;"]):
        count += 1

print(count)

In [ ]:
import re

word_counts = [
    len(re.findall(r"\b\w+\b", p.en))
    for p in paragraphs
]

print(f"Paragraphs : {len(paragraphs)}")
print(f"Average words : {sum(word_counts)/len(word_counts):.2f}")
print(f"Median words : {sorted(word_counts)[len(word_counts)//2]}")
print(f"Max words : {max(word_counts)}")
print(f"Min words : {min(word_counts)}")

In [ ]:
NUM_EXAMPLES = 25

for i, p in enumerate(paragraphs[:NUM_EXAMPLES], 1):

    #print("=" * 120)
    print(f"Paragraph {i}")
    #print("=" * 120)

    print(f"Words   : {len(re.findall(r'\\b\\w+\\b', p.en))}")
    print(f"Samples : {p.num_samples}")
    print(f"Indices : {p.start_index} -> {p.end_index}")
    print()

    print("English")
    #print("-" * 120)
    print(p.en)
    print()

    print("Arabic")
    #print("-" * 120)
    print(p.ar)
    print("\n\n")

In [ ]:
import json
import re

preview = []

for i, p in enumerate(paragraphs[:25], 1):

    preview.append({
        "paragraph_id": i,
        "words": len(re.findall(r"\b\w+\b", p.en)),
        "samples": p.num_samples,
        "start_index": p.start_index,
        "end_index": p.end_index,
        "english": p.en,
        "arabic": p.ar,
    })

with open("paragraph_preview.json", "w", encoding="utf-8") as f:
    json.dump(preview, f, ensure_ascii=False, indent=4)

print("Saved to paragraph_preview.json")

In [ ]:
import re

errors = 0

for i, p in enumerate(paragraphs):

    if not p.en.strip():
        print(f"Empty English at {i}")
        errors += 1

    if not p.ar.strip():
        print(f"Empty Arabic at {i}")
        errors += 1

    if p.start_index > p.end_index:
        print(f"Invalid indices at {i}")
        errors += 1

    if p.num_samples <= 0:
        print(f"Invalid sample count at {i}")
        errors += 1

print(f"\nValidation errors: {errors}")

In [ ]:
import json

OUTPUT_FILE = "untest_paragraphs_ar_en.jsonl"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for p in paragraphs:

        json.dump(
            {
                "translation": {
                    "en": p.en,
                    "ar": p.ar
                }
            },
            f,
            ensure_ascii=False,
        )

        f.write("\n")

print(f"Saved {len(paragraphs)} paragraphs to {OUTPUT_FILE}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    trust_remote_code=True,
)

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    trust_remote_code=True,
)

token_lengths = []

for p in paragraphs:

    text = p.en + "\n" + p.ar

    tokens = tokenizer(
        text,
        add_special_tokens=True,
    )["input_ids"]

    token_lengths.append(len(tokens))

print(f"Paragraphs      : {len(token_lengths)}")
print(f"Average tokens  : {np.mean(token_lengths):.2f}")
print(f"Median tokens   : {np.median(token_lengths):.0f}")
print(f"Max tokens      : {max(token_lengths)}")
print(f"Min tokens      : {min(token_lengths)}")

In [ ]:
print(f"<128    : {sum(x < 128 for x in token_lengths)}")
print(f"128-256 : {sum(128 <= x < 256 for x in token_lengths)}")
print(f"256-512 : {sum(256 <= x < 512 for x in token_lengths)}")
print(f"512-1024: {sum(512 <= x < 1024 for x in token_lengths)}")
print(f">=1024  : {sum(x >= 1024 for x in token_lengths)}")

In [ ]:
MAX_TOKENS = 900

filtered_paragraphs = []
removed = 0

for p in paragraphs:

    text = p.en + "\n" + p.ar

    num_tokens = len(
        tokenizer(
            text,
            add_special_tokens=True,
        )["input_ids"]
    )

    if num_tokens <= MAX_TOKENS:
        filtered_paragraphs.append(p)
    else:
        removed += 1

print(f"Removed   : {removed}")
print(f"Remaining : {len(filtered_paragraphs)}")

paragraphs = filtered_paragraphs

In [ ]:
import json

OUTPUT_FILE = "untest_paragraphs_ar_en_filtered.json"

data = [
    {
        "translation": {
            "en": p.en,
            "ar": p.ar,
        }
    }
    for p in paragraphs
]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Saved {len(data)} paragraphs to {OUTPUT_FILE}")

In [ ]:
import json

OUTPUT_FILE = "untest_paragraphs_ar_en_filtered.jsonl"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for p in paragraphs:

        json.dump(
            {
                "translation": {
                    "en": p.en,
                    "ar": p.ar,
                }
            },
            f,
            ensure_ascii=False,
        )

        f.write("\n")

print(f"Saved {len(paragraphs)} paragraphs to {OUTPUT_FILE}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json

OUTPUT_FILE = "/content/drive/MyDrive/un_paragraphs_ar_en_filtered.jsonl"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for p in paragraphs:

        json.dump(
            {
                "translation": {
                    "en": p.en,
                    "ar": p.ar,
                }
            },
            f,
            ensure_ascii=False,
        )

        f.write("\n")

print(f"Saved {len(paragraphs)} paragraphs to {OUTPUT_FILE}")

In [ ]:
import json
import random

NUM_SAMPLES = 100
OUTPUT_FILE = "/content/un_original_random_100.json"

random.seed(42)

reservoir = []

for i, sample in enumerate(dataset):

    if i < NUM_SAMPLES:
        reservoir.append(sample)
    else:
        j = random.randint(0, i)
        if j < NUM_SAMPLES:
            reservoir[j] = sample

preview = []

for i, sample in enumerate(reservoir, 1):

    preview.append({
        "index": i,
        "english": sample["translation"]["en"],
        "arabic": sample["translation"]["ar"],
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(preview, f, ensure_ascii=False, indent=4)

print(f"Saved {len(preview)} random samples to {OUTPUT_FILE}")

In [ ]:
import json
import random

NUM_SAMPLES = 500
OUTPUT_FILE = "/content/untest500_paragraphs_random_500.json"

random.seed(42)  # For reproducibility

samples = random.sample(paragraphs, min(NUM_SAMPLES, len(paragraphs)))

preview = []

for i, p in enumerate(samples, 1):

    preview.append({
        "paragraph_id": i,
        "num_samples": p.num_samples,
        "start_index": p.start_index,
        "end_index": p.end_index,
        "english": p.en,
        "arabic": p.ar,
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(preview, f, ensure_ascii=False, indent=4)

print(f"Saved {len(preview)} random samples to {OUTPUT_FILE}")